# How AI "Thinks": Building Probability Models Together

**Learning Goals:**
- Understand that AI language models are fundamentally based on probabilities
- Build and use probability distributions from data
- See the connection between simple probability models and sophisticated AI

---

In [ ]:
from datascience import *
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## Part 1: Collective Wisdom - Building Our Probability Distribution

### The Setup

Our class was asked to complete this sentence:

> **"On a cold winter morning, I like to drink ___"**

Everyone submitted their answer. Let's see what we got!

In [ ]:
sheet_id = '1THuKOo5EEkx3F7sIwtcCggLB7LHNdTN1xvJUUouAaZU'
gid = '1983745226'
csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"

In [ ]:
sentence_table = Table.read_table(csv_url)
sentence_table

In [ ]:
drink_data = sentence_table.select('On a cold winter morning, I like to drink ___ ')
drink_data.show(10)

### Building the Probability Distribution

Let's count how many times each response appeared and calculate probabilities:

In [ ]:
# Count responses and calculate probabilities
drink_probs = drink_data.group('On a cold winter morning, I like to drink ___ ')
total_responses = drink_probs.column('count').sum()

drink_probs = drink_probs.with_column(
    'Probability', 
    drink_probs.column('count') / total_responses
)

# Sort by probability for easier reading
drink_probs = drink_probs.sort('Probability', descending=True)
drink_probs

In [ ]:
# Visualize the probability distribution
drink_probs.barh('On a cold winter morning, I like to drink ___ ', 'Probability')
plt.xlabel('Probability')
plt.title('What Our Class Drinks on Cold Winter Mornings')
plt.tight_layout()

### 🤔 Discussion Questions:
1. Which response was most common? Why do you think that is?
2. What's the probability of someone saying "coffee"?
3. What's the probability of someone saying something OTHER than coffee?
4. If we asked 100 new students, about how many would you expect to say "tea"?

# Part II: Building Our Class AI: From Sentences to Generation

**Learning Goals:**
- Build a bigram language model from real sentences
- Understand how training data shapes AI behavior
- See how one-word prompts guide text generation
- Connect bigram probabilities to modern AI prompting

**Prerequisites:** Complete both previous AI probability notebooks!

---

## The Big Idea

In this notebook, we'll:
1. Collect short sentences from everyone in class
2. Build a probability model that learns which words follow which words
3. Use one-word "prompts" to generate new sentences
4. See how our model behaves differently based on the prompt!

This is *exactly* how AI language models work - just scaled up massively!

## Part II A: Our Training Data

### The Exercise

Each student was asked to write **three short sentences** (5-10 words each) on any topic. These sentences are our "training corpus" - the data our AI will learn from!

**Example submissions:**
- "The coffee shop was crowded this morning."
- "I love watching movies on rainy days."
- "My dog chases squirrels in the park."
- "The test was harder than I expected."
- "Students need more sleep during finals week."

In [ ]:
# Simulated student sentences - replace with actual Google Form data!
# In practice: sentences = Table.read_table('class_sentences.csv').column('Sentence')

student_sentences = make_array(
    # Student 1
    'the weather today is absolutely beautiful',
    'students study hard during finals week',
    'my favorite food is definitely pizza',
    # Student 2
    'the library gets crowded before exams',
    'i love listening to music while studying',
    'coffee helps me stay awake late',
    # Student 3
    'the professor explained the concept clearly',
    'my dog loves playing in the park',
    'friends help each other with homework',
    # Student 4
    'the campus looks beautiful in spring',
    'students often eat pizza on weekends',
    'studying late requires lots of coffee',
    # Student 5
    'the weather changes quickly in april',
    'my favorite time is definitely morning',
    'coffee shops are perfect for studying',
    # Student 6
    'the exam was surprisingly easy today',
    'music helps me concentrate better always',
    'pizza tastes best with friends nearby',
    # Student 7
    'the park is crowded on weekends',
    'students need sleep during finals week',
    'my professor is very helpful always'
)

print(f"Total sentences collected: {len(student_sentences)}")
print("\nFirst 10 sentences:")
for i, sentence in enumerate(student_sentences[:10], 1):
    print(f"{i}. {sentence}")

### Our class sentences

[Google Form](https://forms.gle/kbGWVR5HwhPDKxGY7)

In [ ]:
sheet_id = '1THuKOo5EEkx3F7sIwtcCggLB7LHNdTN1xvJUUouAaZU'
gid = '1983745226'
csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"

In [ ]:
sentence_table = Table.read_table(csv_url)
sentence_table

In [ ]:
student_sentences = sentence_table.column('Sentence')

In [ ]:
print(f"Total sentences collected: {len(student_sentences)}")
print("\nFirst 10 sentences:")
for i, sentence in enumerate(student_sentences[:10], 1):
    print(f"{i}. {sentence}")

### 🤔 Quick Observation:
Notice any common words or themes? That's because we all share similar experiences and vocabulary. Real AI training data has the same property - it reflects patterns in how people actually write and speak!

## Part II B: Building the Bigram Model

A **bigram** is a pair of consecutive words. Our model will learn:
- Given word A, what's likely to follow?
- We do this by counting all word pairs in our corpus

In [ ]:
def create_bigrams(sentences):
    """Extract all word pairs from sentences"""
    current_words = []
    next_words = []
    
    for sentence in sentences:
        # Convert to lowercase and split into words
        words = sentence.lower().split()
        
        # Create pairs of consecutive words
        for i in range(len(words) - 1):
            current_words.append(words[i])
            next_words.append(words[i + 1])
    
    return current_words, next_words

# Build the bigram table
current_words, next_words = create_bigrams(student_sentences)

bigrams = Table().with_columns(
    'current_word', current_words,
    'next_word', next_words
)

print(f"Total word pairs (bigrams): {bigrams.num_rows}")
print("\nSample bigrams:")
bigrams.show(15)

### Understanding the Bigram Table

Each row shows a word pair that appeared in our training data:
- **current_word**: A word that appeared
- **next_word**: The word that came immediately after it

This is our "memory" of how words connect!

## Part 3: What Comes After? Conditional Probabilities

Now we can answer: "Given a word, what's likely to come next?"

In [ ]:
def next_word_probabilities(word, bigram_table):
    """
    Calculate P(next_word | current_word)
    Returns a table of possible next words and their probabilities
    """
    word = word.lower()  # Make case-insensitive
    
    # Filter to rows where current_word matches
    following = bigram_table.where('current_word', word)
    
    if following.num_rows == 0:
        return None  # Word not in our training data
    
    # Count occurrences of each next word
    counts = following.group('next_word')
    total = counts.column('count').sum()
    
    # Calculate probabilities
    return counts.with_column(
        'probability',
        counts.column('count') / total
    ).sort('probability', descending=True)

# Let's explore some examples
print("What comes after 'the'?")
next_word_probabilities('the', bigrams)

In [ ]:
print("What comes after 'students'?")
next_word_probabilities('students', bigrams)

In [ ]:
print("What comes after 'coffee'?")
next_word_probabilities('coffee', bigrams)

In [ ]:
print("What comes after 'my'?")
next_word_probabilities('my', bigrams)

### 🤔 Discussion Questions:
1. Why does "the" have so many possible next words?
2. Why are some words more predictable than others?
3. What happens if you ask for a word that wasn't in our training data?
4. How is this like AI "knowing" what words go together?

## Part 4: Text Generation - Our AI in Action!

Now for the exciting part: generating sentences using our trained model!

### The Algorithm:
1. Start with a **prompt word** (user's choice)
2. Look up P(next_word | current_word)
3. Sample a next word from that probability distribution
4. Repeat from step 2 using the new word
5. Stop after N words or when we hit a dead end

In [ ]:
def generate_sentence(start_word, bigram_table, max_words=12):
    """
    Generate a sentence starting with start_word
    Uses the bigram probability model to choose next words
    """
    start_word = start_word.lower()
    sentence = [start_word]
    current = start_word
    
    for _ in range(max_words - 1):
        # Get probability distribution for next word
        probs = next_word_probabilities(current, bigram_table)
        
        if probs is None or probs.num_rows == 0:
            # No known next words - stop here
            break
        
        # Sample next word according to probabilities
        next_options = probs.column('next_word')
        probabilities = probs.column('probability')
        
        next_word = np.random.choice(next_options, p=probabilities)
        sentence.append(next_word)
        current = next_word
    
    return ' '.join(sentence)

# Test it out!
print("Generating sentences with different prompts:\n")

for prompt in ['the', 'students', 'my', 'coffee', 'i']:
    print(f"Prompt: '{prompt}'")
    for i in range(3):
        print(f"  {i+1}. {generate_sentence(prompt, bigrams)}")
    print()

### 🤔 Discussion Questions:
1. Do the generated sentences make grammatical sense?
2. Do they make *semantic* sense (meaning)?
3. Why do you get different sentences each time with the same prompt?
4. How does the choice of starting word affect the output?

## Part 5: Prompt Matters! Comparing Outputs

Just like with ChatGPT, the initial prompt shapes everything that follows. Let's generate multiple sentences from different prompts and see how they differ.

In [ ]:
def compare_prompts(prompts, bigram_table, n_samples=5):
    """
    Generate multiple sentences for each prompt and compare
    """
    for prompt in prompts:
        print(f"\n{'='*60}")
        print(f"PROMPT: '{prompt}'")
        print('='*60)
        
        generations = []
        for i in range(n_samples):
            gen = generate_sentence(prompt, bigram_table)
            generations.append(gen)
            print(f"{i+1}. {gen}")
        
        # Count unique generations
        unique = len(set(generations))
        print(f"\n→ Variety: {unique}/{n_samples} unique sentences")

# Compare different starting words
compare_prompts(['the', 'students', 'coffee', 'my'], bigrams, n_samples=5)

### 🤔 Discussion Questions:
1. Which prompt produces the most varied outputs? Why?
2. Which prompt produces more repetitive outputs? Why?
3. Can you see topics/themes that emerge from certain prompts?
4. How is this like asking ChatGPT different questions?

## Part 6: Analyzing Our Model's "Vocabulary"

Let's explore what words our AI "knows" and how connected they are.

In [ ]:
# Find unique words that can START a generation
starting_words = bigrams.group('current_word')
print(f"Total unique words that can start generation: {starting_words.num_rows}")
print("\nMost common starting words:")
starting_words.sort('count', descending=True).show(15)

In [ ]:
# Visualize most common starting words
top_starters = starting_words.sort('count', descending=True).take(np.arange(10))

plt.figure(figsize=(10, 6))
plt.barh(top_starters.column('current_word'), top_starters.column('count'))
plt.xlabel('Number of times word appears as current_word')
plt.ylabel('Word')
plt.title('Top 10 Words in Our Training Data')
plt.gca().invert_yaxis()
plt.tight_layout()

In [ ]:
# Find words with the MOST options for what comes next
options_count = []
for word in starting_words.column('current_word'):
    probs = next_word_probabilities(word, bigrams)
    options_count.append([word, probs.num_rows])

variety_table = Table(['word', 'num_options']).with_rows(options_count)
variety_table = variety_table.sort('num_options', descending=True)

print("Words with the MOST variety (most different next words):")
variety_table.show(10)

### 🤔 Discussion Questions:
1. Why do common words like "the" and "is" appear so often?
2. Why do they also have the most variety in what comes next?
3. What would happen if we had 1000 sentences instead of 21?
4. How does training data size affect AI capabilities?

## Part 7: Interactive Generation - Try Your Own Prompts!

In [ ]:
def interactive_generation(bigram_table):
    """
    Interactive prompt testing
    """
    print("Available starting words (showing first 20):")
    available = sorted(bigram_table.group('current_word').column('current_word'))
    print(available[:20])
    print(f"\n... and {len(available) - 20} more words\n")
    
    # Example prompts
    test_prompts = ['the', 'students', 'my', 'pizza', 'studying']
    
    print("Try these prompts (run cell multiple times to see variety):\n")
    for prompt in test_prompts:
        if prompt in available:
            print(f"'{prompt}': {generate_sentence(prompt, bigram_table)}")
    
    print("\n" + "="*60)
    print("Want to try your own? Modify test_prompts in the code above!")
    print("="*60)

interactive_generation(bigrams)

## Part 8: The Big Picture - From Our Model to ChatGPT

### What we built:
✅ A probability model trained on text data  
✅ Conditional probabilities: P(word_next | word_current)  
✅ Text generation by sampling from distributions  
✅ Prompt-dependent output  
✅ Vocabulary limited by training data  

### How ChatGPT extends this:

| Our Model | ChatGPT/Claude |
|-----------|----------------|
| 21 sentences | Trillions of words |
| ~100 unique words | Millions of words |
| Looks at 1 previous word | Looks at 1000s of previous words |
| Exact bigram matching | Neural network patterns |
| Simple probability tables | 175+ billion parameters |
| Generates ~5-10 words | Generates full essays |

### But the core principles are IDENTICAL:
1. **Learn from data**: Both learn probability distributions from text
2. **Context matters**: Both use previous words to predict next words
3. **Sampling creates variety**: Both generate different outputs from same prompt
4. **Prompts guide behavior**: Both are steered by how you start them
5. **Training shapes output**: Both reflect patterns in their training data

### Why This Matters:

When you use ChatGPT:
- It's doing exactly what we just did, but at massive scale
- Your prompt shifts probability distributions over billions of possible next words
- Each word is chosen by sampling from those probabilities
- The quality depends on its training data (just like ours!)
- It can't know things not in its training data (just like ours!)

**You just built a real AI language model!** 🎉

## Part 9: Limitations and Insights

### What Our Model Struggles With:

Run these cells to see common failure modes:

In [ ]:
# Problem 1: Out-of-vocabulary words
print("Trying to start with a word NOT in our training data:\n")
test_words = ['quantum', 'basketball', 'algorithm']
for word in test_words:
    result = next_word_probabilities(word, bigrams)
    if result is None:
        print(f"'{word}': Can't generate - word not in training data!")
    else:
        print(f"'{word}': {generate_sentence(word, bigrams)}")

In [ ]:
# Problem 2: Dead ends
print("Words that lead to dead ends (no known next word):\n")
for word in ['week', 'always', 'nearby', 'clearly']:
    probs = next_word_probabilities(word, bigrams)
    if probs is not None:
        print(f"'{word}': {probs.num_rows} options")
        if probs.num_rows == 0:
            print(f"  → DEAD END! This word never has a next word in our data")

In [ ]:
# Problem 3: Repetitive patterns
print("Watch out for loops and repetition:\n")
for i in range(5):
    sentence = generate_sentence('the', bigrams, max_words=20)
    words = sentence.split()
    if len(words) != len(set(words)):
        print(f"{i+1}. {sentence} ⚠️ (has repeated words!)")
    else:
        print(f"{i+1}. {sentence}")

### 🤔 Discussion Questions:
1. Why can't our model use words it wasn't trained on?
2. Real AI has the same limitation - why do we need such massive training data?
3. What causes dead ends in generation?
4. How might loops/repetition happen? How does real AI avoid this?

## 🎯 Key Takeaways

1. **AI = Learned Probabilities**: Language models learn probability distributions from training data

2. **Context is Everything**: Previous words determine the probability distribution over next words

3. **Prompts Matter**: Starting words dramatically affect generated output

4. **Training Data Shapes Behavior**: Our AI can only work with patterns it's seen

5. **Sampling Creates Variety**: Random sampling from probabilities means different outputs each time

6. **Scale Makes the Magic**: The difference between our model and ChatGPT is scale, not fundamental approach

---

## 🏠 Extension Activities

1. **Collect more data**: Gather 100 sentences instead of 21. How does performance improve?

2. **Themed training**: Have everyone write sentences about ONE topic (sports, food, tech). How does this affect outputs?

3. **Trigrams**: Extend to 3-word sequences. P(word₃ | word₁, word₂). More realistic?

4. **Temperature**: Add a temperature parameter to make sampling more/less random

5. **Beam search**: Instead of random sampling, always pick the most likely word. What happens?

6. **Compare to real AI**: Take the same prompts to ChatGPT. How do outputs differ?

7. **Sentence starters**: Analyze which words make good vs bad starting prompts